# 监控与可观测性 第2周:Grafana — 可视化与告警

> **学习目标**:能用 Grafana 创建 Dashboard、配置告警规则、设计 SLO Dashboard

---

## Day 8-9:Grafana Dashboard 设计原则

一个良好的 Dashboard 应该"从上到下,从左到右"讲述一个故事:

| 行 | 内容 | 推荐 Panel |
|------|------|------------|
| 概览行 | 关键指标(QPS / p99 / error rate / 在线实例数) | Stat panel |
| 趋势行 | 各指标的时间序列曲线 | Time Series panel |
| 明细行 | 按 endpoint / status_code / instance 拆分 | Table 或多个 Time Series |
| 资源行 | CPU / Mem / Disk / Network | Gauge + Time Series |

红色警报原则:
- 用红/黄色表示异常,绿色表示正常
- 单位要明确 (ms vs s, MB vs GB)
- 使用 Variables (`$instance`, `$endpoint`) 实现交互

## Day 10:常用 Panel 类型

| Panel | 用途 |
|-------|------|
| Time Series | 折线图,展示指标随时间变化 → 最常用 |
| Stat | 单个大数字(加 sparkline)→ Dashboard 顶部 |
| Gauge | 仪表盘,显示用量百分比 → CPU/Mem 使用率 |
| Bar Gauge | 横向条形图 → 各实例对比 |
| Table | 表格明细 → Top N 最慢 endpoints |
| Heatmap | Histogram 热力图 → 看延迟分布 |

## Day 11:告警规则设计

In [ ]:
alerts = [
    {
        "alert": "HighErrorRate",
        "expr": 'sum(rate(http_requests_total{status_code=~"5.."}[5m])) by (service) > 0.05',
        "for": "5m",
        "severity": "critical",
        "runbook": "1. 检查最近部署; 2. 查看错误日志; 3. 检查 DB 连接池; 4. 检查依赖服务",
    },
    {
        "alert": "HighLatency",
        "expr": "histogram_quantile(0.99, rate(http_request_duration_seconds_bucket[5m])) > 1",
        "for": "10m",
        "severity": "warning",
        "runbook": "1. 查看 slow query log; 2. 检查 CPU/内存; 3. 检查 GC 尖刺; 4. 查看 Trace",
    },
    {
        "alert": "InstanceDown",
        "expr": "up == 0",
        "for": "2m",
        "severity": "critical",
        "runbook": "1. SSH 到目标机器; 2. 查看进程是否存在; 3. 查看系统日志; 4. K8s Pod 则 describe",
    },
]

for a in alerts:
    print(f"\n{a['alert']} ({a['severity']})")
    print(f"  expr: {a['expr']}")
    print(f"  for: {a['for']}")
    print(f"  runbook: {a['runbook']}")

print()
print("告警分级:")
print("  Critical → 立即处理(On-call 收到通知,5 分钟内响应)")
print("  Warning  → 工作时间处理")
print("  Info     → 不通知,仅 Dashboard 展示")

## Day 12:Alertmanager 路由

Alertmanager 负责三件事:

- **分组(Grouping)**:同类告警合并成一条通知,避免轰炸
- **抑制(Inhibition)**:如果主机宕了,抑制该主机上所有服务告警
- **静默(Silence)**:已知维护窗口内不通知

路由策略:Critical → 飞书 + PagerDuty(立即),Warning → 飞书(工作时间),同类告警 5 分钟合并

## Day 13:Recording Rules

预计算常用查询,加速 Dashboard 渲染:

```yaml
rules:
  - record: job:http_requests:rate5m
    expr: rate(http_requests_total[5m])
  - record: job:http_errors:rate5m
    expr: rate(http_requests_total{status_code=~"5.."}[5m])
  - record: job:request_duration:p99
    expr: histogram_quantile(0.99, rate(http_request_duration_seconds_bucket[5m]))
```

好处:Dashboard 加载更快,多个 Dashboard 共享同一个预计算指标

## Day 14:第2周综合练习

In [ ]:
print("=" * 60)
print("第2周综合练习交付清单")
print("=" * 60)

print("""
Dashboard 1: 服务概览
  - 顶部 4 个 Stat panel: QPS / p99延迟 / 错误率 / 在线实例数
  - 中部 Time Series: QPS 趋势 + 延迟趋势
  - 底部 Table: 按 endpoint 的 QPS 和延迟明细

Dashboard 2: 资源监控
  - CPU 使用率 (Gauge)
  - 内存使用率 (Gauge)
  - 磁盘使用率 (Gauge)

告警规则 (4条):
  - HighErrorRate (critical, 5min)
  - HighLatency (warning, 10min)
  - InstanceDown (critical, 2min)
  - DiskFull (warning, 5min)

Alertmanager 配置:
  - Critical → 飞书 (立即)
  - Warning → 飞书 (工作时间)
  - 同类告警 5 分钟合并
  - 机器宕机抑制该机的其他告警

所有 Dashboard 导出 JSON,提交到 Git
""")

print("=" * 60)
print("第2周核心收获:")
print("1. Dashboard 从概览到明细,从上到下讲故事")
print("2. 告警要有 severity 分级、runbook 和合适的 for 时间")
print("3. Alertmanager 负责路由、分组、抑制、静默")
print("4. Recording Rules 预计算,加速 Dashboard 和告警规则")
print("=" * 60)